# Speech Transcription, Word Alignment & Transcript Re-alignment

One facade — `SpeechAlignmentPipeline` — hides the whisperX / forced-alignment /
fuzzy-matching machinery and returns **structured dataclasses** you can act on
(e.g. drive audio cutting in another project).

| Task | Method | Structured output |
|---|---|---|
| 1. Audio → transcript | `SpeechAlignmentPipeline.transcribe` | `str` |
| 2. Audio → word timestamps | `SpeechAlignmentPipeline.transcribe_words` | `list[Word]` |
| 3. Check an annotation | `TranscriptEvaluator.evaluate` (or `pipe.evaluate_annotation`) | `TranscriptMetrics` + `is_acceptable` |
| 4. Validate / re-segment | `SpeechAlignmentPipeline.validate_segments` | `list[SegmentValidation]` |

**Under the hood** (you never call these directly): `WhisperWrapper` (ASR) →
`TorchaudioForcedAligner` or `WhisperxForcedAligner` (word times) →
`find_segment` (rapidfuzz) for re-alignment; metrics use the standard Whisper
normalizer + `xml-roberta` for semantic similarity.

**Dataclasses returned**

| Type | Fields |
|---|---|
| `Word` | `text, start, end, score` |
| `TranscriptMetrics` | `wer, cer, similarity, normalized_wer, normalized_cer, normalized_similarity, semantic_similarity, reference, hypothesis` |
| `SegmentMatch` | `text, start, end, score, word_start_idx, word_end_idx` |
| `SegmentValidation` | `text, annotated_start/end, recovered_start/end, start/end_offset, score, decision, match` |

Demo audio: `fi_protagonist.wav` (a YouTube-style monologue).

In [1]:
from pathlib import Path

from exordium.audio.io import load_audio

audio_path = Path("../tests/fixtures/audio/fi_protagonist.wav")
waveform, sr = load_audio(audio_path, target_sample_rate=16000)
print(f"Audio file : {audio_path}  (exists={audio_path.exists()})")
print(f"Duration   : {waveform.shape[-1] / sr:.1f} s @ {sr} Hz mono")

Audio file : ../tests/fixtures/audio/fi_protagonist.wav  (exists=True)
Duration   : 15.3 s @ 16000 Hz mono


## 0. Build the pipeline (backend hidden behind one object)

In [2]:
from exordium.text.pipeline import SpeechAlignmentPipeline

# backend defaults to "whisperx" (wav2vec2 alignment); "torchaudio" (MMS_FA) also
# available. device_id: -1/None -> CPU, 0+ -> GPU/MPS.
pipe = SpeechAlignmentPipeline(language="en", device_id=-1)
print("pipeline ready — backend:", pipe.backend)

/Volumes/UGREEN/Dev/exordium/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 539/539 [00:01<00:00, 482.88it/s]


pipeline ready — backend: whisperx


## 1. Audio → transcript

`transcribe` returns a plain string (this is `WhisperWrapper` underneath).

In [3]:
transcript = pipe.transcribe(audio_path)
print(f"{type(transcript).__name__}, {len(transcript)} chars (full text below):\n\n{transcript}")

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

str, 226 chars (full text below):

video is going to be a quick one but it's going to be a great one for you guys so let's get started so yes as you can see from the title I hit 10,000 subscribers and you guys just don't know how much that means to me I am very


## 2. Audio → word-level timestamps

`transcribe_words` runs ASR then forced alignment and returns `list[Word]`,
each carrying `start` / `end` (seconds) and an alignment `score`.

In [4]:
words = pipe.transcribe_words(audio_path)
print(f"{len(words)} words. First 12 (each is a Word dataclass):\n")
print(f"{'word':<14}{'start':>8}{'end':>8}{'score':>8}")
for w in words[:12]:
    print(f"{w.text:<14}{w.start:>8.2f}{w.end:>8.2f}{w.score:>8.2f}")

51 words. First 12 (each is a Word dataclass):

word             start     end   score
video             0.00    0.72    0.57
is                0.78    0.86    0.56
going             0.94    1.16    0.85
to                1.20    1.26    0.94
be                1.32    1.48    0.90
a                 1.50    1.52    1.00
quick             1.86    2.13    0.98
one               2.37    2.49    0.90
but               2.83    2.95    0.69
it's              2.99    3.07    0.44
going             3.11    3.23    0.62
to                3.27    3.33    0.87


## 3. Check an annotation against the prediction

`TranscriptEvaluator` compares two strings — a reference (annotation) and a
hypothesis (prediction) — and returns `TranscriptMetrics`. It is **independent of
audio, ASR, and the pipeline** (loads only a sentence-embedding model), so you can
call it on any two strings. *(The pipeline also offers
`pipe.evaluate_annotation(audio, text)`, which transcribes then calls this.)*

Metric views (WER/CER: lower = better, 0 = identical; sim/semantic: higher = better):

* **raw** — lightly normalized; keeps apostrophes/digits, so `"ten thousand"` vs
  `"10,000"` count as errors. Inflated by formatting.
* **normalized** — via **Whisper's standard `EnglishTextNormalizer`** (from
  `transformers`; the Whisper paper / HF Open ASR Leaderboard routine — not
  hand-rolled). Canonicalizes numbers/contractions/casing. **Judge on this.**
* **sim** (0–100) — `rapidfuzz` character-level string similarity (lexical, no model).
* **semantic** (−1…1) — cosine of **xml-roberta** sentence embeddings (meaning-level,
  robust to paraphrase).

First a **sanity check**: feeding the *same* string as both sides must be a perfect
match (0 WER, semantic 1.0).

In [5]:
from exordium.text.evaluation import TranscriptEvaluator, is_acceptable

evaluator = TranscriptEvaluator()  # xml-roberta semantic + standard normalizer

# SANITY CHECK: identical strings MUST be a perfect match.
sanity = evaluator.evaluate(transcript, transcript)
print("sanity — transcript vs itself (identical):")
print(
    f"  normalized : wer={sanity.normalized_wer:.3f}  cer={sanity.normalized_cer:.3f}  "
    f"sim={sanity.normalized_similarity:.1f}"
)
print(f"  semantic   : cosine={sanity.semantic_similarity:.3f}")
assert sanity.normalized_wer == 0.0 and sanity.semantic_similarity > 0.999
print("  -> perfect match (0 WER, semantic 1.0), as expected ✓")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8337.16it/s]


sanity — transcript vs itself (identical):
  normalized : wer=0.000  cer=0.000  sim=100.0
  semantic   : cosine=1.000
  -> perfect match (0 WER, semantic 1.0), as expected ✓


### Accept rule & defaults

`is_acceptable(metrics)` gives a heuristic verdict — accept if **either**:

* `normalized_wer ≤ 0.20` (lexically close — the standard word-level criterion), **or**
* `semantic_similarity ≥ 0.70` (clearly related/paraphrase — rescues high-WER but
  same-meaning pairs).

Both are keyword-overridable defaults (`DEFAULT_MAX_NORMALIZED_WER`,
`DEFAULT_MIN_SEMANTIC_SIMILARITY` in `exordium.text.evaluation`) — there is no
universal ASR cutoff, so tune per dataset (e.g. raise WER to ~0.35 for "fair").
CER and the raw metrics are reported for insight but not part of the decision.

Now the **real** comparison — your annotation vs the prediction:

In [6]:
# The transcript we believe is in the clip (the "annotation") — written casually
# with a spelled-out number, as annotations often are.
annotated = (
    "todays video is going to be a quick one, but it gonna be a great one for you "
    "guys, so lets get started. So yes, as you can see from the title, I hit ten "
    "thousand subscribers and you guys just dont know how much that means to me. Im very"
)

m = evaluator.evaluate(annotated, transcript)  # reference (annotation) vs hypothesis (prediction)
print("real — annotation vs prediction:")
print(f"  raw        : wer={m.wer:.3f}  cer={m.cer:.3f}  sim={m.similarity:.1f}")
print(
    f"  normalized : wer={m.normalized_wer:.3f}  cer={m.normalized_cer:.3f}  "
    f"sim={m.normalized_similarity:.1f}"
)
print(f"  semantic   : cosine={m.semantic_similarity:.3f}  (meaning-level)")
print(f"  is_acceptable -> {is_acceptable(m)}")

real — annotation vs prediction:
  raw        : wer=0.196  cer=0.126  sim=92.1
  normalized : wer=0.157  cer=0.071  sim=96.5
  semantic   : cosine=0.902  (meaning-level)
  is_acceptable -> True


## 4. Validate annotated segments & recover cut points

You have utterance-level annotations `(text, start, end)` and want to check each
one sits where the annotation claims — and, if not, get the corrected times.
`validate_segments` runs **one** ASR pass, then fuzzy-locates every segment.

Each result is a `SegmentValidation` with a `decision`:

* `accept` — annotation within `tolerance` of the true span,
* `recut`  — text found but shifted → use `recovered_start` / `recovered_end`,
* `drop`   — text not found (likely a wrong edit).

In [7]:
# We locate a real phrase to build a KNOWN-GOOD annotation (guaranteed accept),
# then add a deliberately shifted one (recut) and an absent one (drop).
from exordium.text.transcript_align import find_segment

phrase = "so lets get started"
good = find_segment(phrase, words)
print(f"located {phrase!r} at {good.start:.2f}-{good.end:.2f}s (score {good.score:.0f})\n")

segments = [
    # (text, annotated_start, annotated_end)
    (phrase, good.start, good.end),  # -> accept (annotation matches)
    (phrase, good.start + 20.0, good.end + 20.0),  # -> recut  (annotation shifted +20s)
    ("please smash that subscribe button right now", 1.0, 4.0),  # -> drop (not spoken)
]

results = pipe.validate_segments(audio_path, segments, tolerance=0.3)

for v in results:
    print(
        f"decision = {v.decision.upper():7}  score={v.score:5.1f}  "
        f"annotated=({v.annotated_start:.2f},{v.annotated_end:.2f})  "
        f"recovered={None if v.recovered_start is None else f'({v.recovered_start:.2f},{v.recovered_end:.2f})'}"
    )
    if v.decision == "recut":
        print(
            f"           -> cut here: start={v.recovered_start:.2f}s end={v.recovered_end:.2f}s "
            f"(offset {v.start_offset:+.2f}s)"
        )
    print(f"           text: {v.text!r}")

located 'so lets get started' at 5.47-6.74s (score 95)

decision = ACCEPT   score= 94.7  annotated=(5.47,6.74)  recovered=(5.47,6.74)
           text: 'so lets get started'
decision = RECUT    score= 94.7  annotated=(25.48,26.74)  recovered=(5.47,6.74)
           -> cut here: start=5.47s end=6.74s (offset -20.00s)
           text: 'so lets get started'
decision = DROP     score=  0.0  annotated=(1.00,4.00)  recovered=None
           text: 'please smash that subscribe button right now'


### Reading the structured result

`recovered_start` / `recovered_end` are the corrected cut points — hand them to
your own audio-cutting step whenever `decision != "drop"`. Everything is a plain
dataclass, so it serializes cleanly (`dataclasses.asdict`) for logging or JSON.

In [8]:
import json
from dataclasses import asdict

# 'match' holds the underlying SegmentMatch; drop it for a clean JSON-able view.
clean = {k: v for k, v in asdict(results[1]).items() if k != "match"}
print(json.dumps(clean, indent=2))

{
  "text": "so lets get started",
  "annotated_start": 25.475,
  "annotated_end": 26.738,
  "recovered_start": 5.475,
  "recovered_end": 6.738,
  "start_offset": -20.0,
  "end_offset": -20.0,
  "score": 94.73684210526316,
  "decision": "recut"
}


## 5. Swap the alignment backend — same API, torchaudio word times

The default backend above is **whisperX**. Switching to `backend="torchaudio"`
(``MMS_FA``) uses a different forced aligner but returns an identical
`list[Word]`, so every method works unchanged — handy for cross-checking timings.

We pass `whisper=pipe.whisper` so the second pipeline **reuses** the already
-loaded Whisper model instead of loading a second ~1.5 GB copy (Whisper is by
far the largest model here — this keeps memory in check).

In [9]:
pipe_ta = SpeechAlignmentPipeline(
    backend="torchaudio", language="en", device_id=-1, whisper=pipe.whisper
)
words_ta = pipe_ta.transcribe_words(audio_path)

print(f"{'word':<14}{'whisperX':>22}{'torchaudio':>22}")
for a, b in list(zip(words, words_ta))[:8]:
    print(f"{a.text:<14}{f'{a.start:.2f}-{a.end:.2f}':>22}{f'{b.start:.2f}-{b.end:.2f}':>22}")

word                        whisperX            torchaudio
video                      0.00-0.72             0.18-0.52
is                         0.78-0.86             0.74-0.84
going                      0.94-1.16             0.90-1.12
to                         1.20-1.26             1.16-1.24
be                         1.32-1.48             1.28-1.36
a                          1.50-1.52             1.56-1.58
quick                      1.86-2.13             1.74-2.06
one                        2.37-2.49             2.26-2.48
